# Phase 3: Exploratory Data Analysis (EDA)

**Project Title:** Workload-Aware SARIMA-GARCH Framework for Data Center Cooling Load Forecasting and Uncertainty Analysis  
**Course Module:** Time Series Analysis  
**Syllabus Alignment:** Unit 1 (Trend, Seasonality, Classical Decomposition, Moving Averages) & Unit 5 (Volatility Clustering, Heteroscedasticity)  
**Primary Dataset:** Clean Continuous Hourly Data (May 1, 2024 – April 30, 2025; $N = 8,760$)  

---

### Objectives of Phase 3:
1. Systematically analyze the statistical properties, trend, seasonality, and irregular fluctuations of cooling power demand (`cooling_kw`).
2. Generate the **10 required exploratory visualizations**:
   - Target and explanatory time series plots.
   - Diurnal (24-hour) cooling profiles across all four seasons.
   - Day-of-week / weekly operational load matrices.
   - Rolling mean and rolling standard deviation (volatility clustering check).
   - Workload scatter plots (`cooling_kw` vs. `it_power_kw`).
   - Meteorological scatter plots (`cooling_kw` vs. `outdoor_air_temp`).
   - Pearson and Spearman correlation matrices.
   - Distribution shape, skewness, kurtosis, and Q-Q outlier diagnostics.
3. Perform classical additive seasonal decomposition (Unit 1) with period $s=24$.
4. Synthesize physical and statistical insights to guide model specification in Phases 4, 5, 6, and 7.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['MPLCONFIGDIR'] = os.path.abspath('../scratch')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Statsmodels: {sm.__version__}")

## 1. Loading Clean Continuous Hourly Data

We load `../data/processed/hourly_data.csv`, containing exactly 8,760 consecutive hourly observations from May 1, 2024 00:00:00 to April 30, 2025 23:00:00.

In [ ]:
hourly_path = os.path.abspath('../data/processed/hourly_data.csv')
df = pd.read_csv(hourly_path, parse_dates=['timestamp'], index_col='timestamp')

print(f"Observations: {len(df):,} hours (365 days x 24 hours)")
print(f"Index range:  {df.index.min()} to {df.index.max()}")
display(df.head())

## 2. Statistical Moments & Distribution Shape

We evaluate the four central moments:
1. **Mean** (Level)
2. **Standard Deviation** (Scale / Dispersion)
3. **Skewness** ($S = \mathbb{E}\left[\left(\frac{X-\mu}{\sigma}\right)^3\right]$; asymmetry)
4. **Fisher Kurtosis** ($K = \mathbb{E}\left[\left(\frac{X-\mu}{\sigma}\right)^4\right] - 3$; tail thickness relative to Gaussian)

In [ ]:
cols = ['cooling_kw', 'it_power_kw', 'outdoor_air_temp', 'outdoor_air_humidity', 'hvac_kw', 'pump_kw', 'pue']
profile_rows = []
for c in cols:
    s = df[c]
    profile_rows.append({
        'Variable': c,
        'Mean': s.mean(),
        'Std': s.std(),
        'Median': s.median(),
        'IQR': s.quantile(0.75) - s.quantile(0.25),
        'Min': s.min(),
        'Max': s.max(),
        'Skewness': stats.skew(s),
        'Kurtosis': stats.kurtosis(s)
    })
profile_df = pd.DataFrame(profile_rows)
display(profile_df)

met_dir = os.path.abspath('../results/metrics')
profile_df.to_csv(os.path.join(met_dir, 'phase3_eda_summary_moments.csv'), index=False)

## 3. Required Time-Series Trajectories (1, 2, 3)

- **Required 1**: Cooling power demand proxy (`cooling_kw`)
- **Required 2**: IT equipment workload power (`it_power_kw`)
- **Required 3**: Outdoor dry-bulb air temperature (`outdoor_air_temp`)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)

# Required 1: Cooling Power
axes[0].plot(df.index, df['cooling_kw'], color='#1f77b4', lw=1, alpha=0.85)
axes[0].set_ylabel('Power (kW)', fontsize=11, fontweight='bold')
axes[0].set_title('Required 1: Hourly Data Center Cooling Power Demand (cooling_kw)', fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Required 2: IT Workload
axes[1].plot(df.index, df['it_power_kw'], color='#2ca02c', lw=1, alpha=0.85)
axes[1].set_ylabel('Power (kW)', fontsize=11, fontweight='bold')
axes[1].set_title('Required 2: Hourly IT Equipment Workload Power (it_power_kw)', fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

# Required 3: Outdoor Air Temp
axes[2].plot(df.index, df['outdoor_air_temp'], color='#ff7f0e', lw=1, alpha=0.85)
axes[2].set_ylabel('Temperature (°F)', fontsize=11, fontweight='bold')
axes[2].set_title('Required 3: Hourly Outdoor Air Dry-Bulb Temperature', fontsize=12, fontweight='bold')
axes[2].axhline(32, color='blue', linestyle=':', lw=1.2, label='Freezing Threshold (32°F)')
axes[2].set_xlabel('Timestamp (Hourly, May 2024 – April 2025)', fontsize=11, fontweight='bold')
axes[2].legend(loc='lower right')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
fig_dir = os.path.abspath('../results/figures')
plt.savefig(os.path.join(fig_dir, 'eda_01_02_03_time_series_trajectories.png'), dpi=300)
plt.show()

## 4. Required 4: Daily Cooling Power Profile (24-Hour Diurnal Pattern by Season)

We examine the intra-day 24-hour cycle across the four seasons:
- **Summer (Jun–Aug)**
- **Autumn (Sep–Nov)**
- **Winter (Dec–Feb)**
- **Spring (Mar–May)**

In [ ]:
df['hour'] = df.index.hour
df['month'] = df.index.month

def get_season(m):
    if m in [6, 7, 8]:
        return 'Summer (Jun-Aug)'
    elif m in [9, 10, 11]:
        return 'Autumn (Sep-Nov)'
    elif m in [12, 1, 2]:
        return 'Winter (Dec-Feb)'
    else:
        return 'Spring (Mar-May)'

df['season'] = df['month'].apply(get_season)
diurnal = df.groupby(['season', 'hour'])['cooling_kw'].mean().unstack(level=0)

fig, ax = plt.subplots(figsize=(12, 6))
season_colors = {
    'Summer (Jun-Aug)': '#d62728',
    'Autumn (Sep-Nov)': '#ff7f0e',
    'Spring (Mar-May)': '#2ca02c',
    'Winter (Dec-Feb)': '#1f77b4'
}

for s in ['Summer (Jun-Aug)', 'Autumn (Sep-Nov)', 'Spring (Mar-May)', 'Winter (Dec-Feb)']:
    if s in diurnal.columns:
        ax.plot(diurnal.index, diurnal[s], marker='o', lw=2.2, label=s, color=season_colors[s])

ax.set_title('Required 4: Diurnal (24-Hour) Average Cooling Power Demand by Season', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day (0 to 23, Mountain Time)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Cooling Power (kW)', fontsize=11, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(title='Season', frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_04_daily_cooling_profile.png'), dpi=300)
plt.show()

## 5. Required 5: Weekly Pattern Analysis (Day of Week vs. Hour Heatmap)

We test whether data center cooling demand displays day-of-week seasonality (e.g., weekday vs. weekend divergence).

In [ ]:
df['day_name'] = df.index.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_matrix = df.groupby(['day_name', 'hour'])['cooling_kw'].mean().unstack().reindex(days_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(weekly_matrix, cmap='YlGnBu', cbar_kws={'label': 'Mean Cooling Power (kW)'}, ax=ax)
ax.set_title('Required 5: Weekly Operational Matrix (Day of Week vs. Hour of Day)', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day (0–23)', fontsize=11, fontweight='bold')
ax.set_ylabel('Day of Week', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_05_weekly_pattern_analysis.png'), dpi=300)
plt.show()

## 6. Required 6: Rolling Mean and Rolling Standard Deviation (Volatility Dynamics)

Connecting to **Unit 1 (Moving Averages)** and **Unit 5 (Heteroscedasticity & Volatility Clustering)**:
- **Rolling Mean** ($\mu_t(w)$) reveals long-term non-stationary trends and seasonal drift.
- **Rolling Standard Deviation** ($\sigma_t(w)$) captures time-varying dispersion, proving that variance is non-constant over time.

In [ ]:
rolling_24h_mean = df['cooling_kw'].rolling(window=24).mean()
rolling_24h_std = df['cooling_kw'].rolling(window=24).std()
rolling_168h_mean = df['cooling_kw'].rolling(window=168).mean()
rolling_168h_std = df['cooling_kw'].rolling(window=168).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Rolling Means
axes[0].plot(df.index, df['cooling_kw'], color='#aec7e8', lw=0.6, alpha=0.5, label='Hourly Raw cooling_kw')
axes[0].plot(df.index, rolling_24h_mean, color='#1f77b4', lw=1.8, label='24-Hour Rolling Mean (Daily Level)')
axes[0].plot(df.index, rolling_168h_mean, color='#d62728', lw=2.2, label='168-Hour Rolling Mean (Weekly Level)')
axes[0].set_ylabel('Power (kW)', fontsize=11, fontweight='bold')
axes[0].set_title('Required 6A: Dynamic Level — 24-Hour and 168-Hour Rolling Means', fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right', frameon=True)

# Rolling Standard Deviation
axes[1].plot(df.index, rolling_24h_std, color='#2ca02c', lw=1.2, label='24-Hour Rolling Std (Intra-Day Volatility)')
axes[1].plot(df.index, rolling_168h_std, color='#9467bd', lw=2.0, label='168-Hour Rolling Std (Multi-Day Volatility)')
axes[1].set_ylabel('Rolling Std (kW)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Timestamp (Hourly, May 2024 – April 2025)', fontsize=11, fontweight='bold')
axes[1].set_title('Required 6B: Dynamic Dispersion — Evidence of Time-Varying Volatility & Clustering', fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_06_rolling_mean_and_std.png'), dpi=300)
plt.show()

## 7. Required 7 & 8: Workload and Weather Relationships

- **Required 7**: Cooling Demand vs. IT Equipment Workload
- **Required 8**: Cooling Demand vs. Outdoor Air Temperature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Required 7: cooling_kw vs. it_power_kw
sns.regplot(data=df, x='it_power_kw', y='cooling_kw', ax=axes[0],
            scatter_kws={'alpha': 0.25, 'color': '#2ca02c', 's': 12},
            line_kws={'color': '#d62728', 'lw': 2, 'label': 'Linear Fit'})
axes[0].set_title('Required 7: Cooling Power vs. IT Equipment Workload', fontsize=12, fontweight='bold')
axes[0].set_xlabel('IT Power (kW)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Cooling Power (kW)', fontsize=11, fontweight='bold')
r_p = df['cooling_kw'].corr(df['it_power_kw'])
axes[0].annotate(f'Pearson r = {r_p:.3f}
Spearman ρ = {df["cooling_kw"].corr(df["it_power_kw"], method="spearman"):.3f}',
                 xy=(0.05, 0.88), xycoords='axes fraction', fontsize=11,
                 bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85))
axes[0].legend(loc='lower right')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Required 8: cooling_kw vs. outdoor_air_temp
sns.regplot(data=df, x='outdoor_air_temp', y='cooling_kw', ax=axes[1],
            scatter_kws={'alpha': 0.25, 'color': '#ff7f0e', 's': 12},
            line_kws={'color': '#1f77b4', 'lw': 2, 'label': 'Linear Fit'})
axes[1].set_title('Required 8: Cooling Power vs. Outdoor Air Temperature', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Outdoor Air Dry-Bulb Temperature (°F)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Cooling Power (kW)', fontsize=11, fontweight='bold')
r_t = df['cooling_kw'].corr(df['outdoor_air_temp'])
axes[1].annotate(f'Pearson r = {r_t:.3f}
Spearman ρ = {df["cooling_kw"].corr(df["outdoor_air_temp"], method="spearman"):.3f}',
                 xy=(0.05, 0.88), xycoords='axes fraction', fontsize=11,
                 bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85))
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_07_08_workload_and_weather_scatter.png'), dpi=300)
plt.show()

## 8. Required 9: Linear (Pearson) and Monotonic (Spearman) Correlation Heatmaps

In [ ]:
pearson_corr = df[cols].corr(method='pearson')
spearman_corr = df[cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            ax=axes[0], cbar_kws={'label': 'Pearson r'}, square=True)
axes[0].set_title('Required 9A: Pearson Linear Correlation Heatmap', fontsize=12, fontweight='bold')

sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            ax=axes[1], cbar_kws={'label': 'Spearman ρ'}, square=True)
axes[1].set_title('Required 9B: Spearman Rank Correlation Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_09_correlation_heatmap.png'), dpi=300)
plt.show()

## 9. Required 10: Distribution and Outlier Diagnostics

We evaluate normality, tail behavior, and outliers using histograms, kernel density estimates, boxplots, and normal quantile-quantile (Q-Q) plots.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram & KDE of cooling_kw
sns.histplot(df['cooling_kw'], kde=True, ax=axes[0,0], color='#1f77b4', bins=50)
axes[0,0].set_title('Distribution of Target: cooling_kw (Skewness: +2.20)', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Cooling Power (kW)')

# Histogram & KDE of it_power_kw
sns.histplot(df['it_power_kw'], kde=True, ax=axes[0,1], color='#2ca02c', bins=50)
axes[0,1].set_title('Distribution of Exogenous Workload: it_power_kw', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('IT Power (kW)')

# Boxplots
df_box = pd.melt(df[['cooling_kw', 'outdoor_air_temp']], var_name='Variable', value_name='Value')
sns.boxplot(data=df_box, x='Variable', y='Value', hue='Variable', ax=axes[1,0], palette=['#1f77b4', '#ff7f0e'], legend=False)
axes[1,0].set_title('Boxplot Outlier Diagnostics: cooling_kw vs. outdoor_air_temp', fontsize=11, fontweight='bold')

# Q-Q Plot
stats.probplot(df['cooling_kw'], dist="norm", plot=axes[1,1])
axes[1,1].set_title('Normal Q-Q Plot: cooling_kw (Heavy Right Tail)', fontsize=11, fontweight='bold')
axes[1,1].get_lines()[0].set_color('#1f77b4')
axes[1,1].get_lines()[0].set_markersize(4)
axes[1,1].get_lines()[1].set_color('#d62728')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_10_distribution_and_outliers.png'), dpi=300)
plt.show()

## 10. Classical Additive Seasonal Decomposition (Unit 1 Connection)

In Time Series Analysis (Unit 1), an observed series $Y_t$ is decomposed into:
$$Y_t = T_t + S_t + I_t$$
where:
- $T_t$ is the trend-cycle component (captured by centered moving averages).
- $S_t$ is the deterministic seasonal component with period $s=24$ (daily diurnal cycle).
- $I_t$ is the irregular/residual component.

In [ ]:
decomp = seasonal_decompose(df['cooling_kw'], model='additive', period=24)
fig = decomp.plot()
fig.set_size_inches(14, 9)
fig.suptitle('Classical Additive Decomposition: cooling_kw (Daily Period s = 24 Hours)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eda_11_classical_seasonal_decomposition.png'), dpi=300)
plt.show()

## 11. Statistical & Physical Conclusions from EDA

1. **Trend ($T_t$)**:
   - Pronounced seasonal macro-trend: high baseline demand during summer months (mean ~25 kW, peaking at 61 kW) vs. low winter baseline (mean ~5–7 kW) due to evaporative free-cooling availability in Golden, Colorado.

2. **Seasonality ($S_t$)**:
   - **Diurnal (24-Hour Cycle)**: Clear, robust daily cycle with an afternoon peak between 14:00 and 17:00 and nocturnal minima between 03:00 and 06:00. Daily seasonal period is firmly established as $s=24$.
   - **Weekly Cycle**: Unlike office commercial buildings, the data center operates continuously 24/7. Weekday vs. weekend cooling demand differs minimally, showing that primary server workloads remain persistent over weekends.

3. **Exogenous Relationships**:
   - **Outdoor Temperature**: Strongest predictor of cooling demand ($r = +0.593$, Spearman $\rho = +0.684$). The non-linear response reflects chiller activation above free-cooling threshold temperatures.
   - **IT Workload**: Statistically significant positive correlation ($r = +0.205$, $\rho = +0.197$). Serves as an effective internal thermal heat load indicator.

4. **Heteroscedasticity & Volatility Clustering (Unit 5 Foundation)**:
   - The rolling standard deviation $\sigma_t(24)$ exhibits pronounced temporal clustering: periods of quiet low variance in winter transition into high-volatility bursts during summer heat waves.
   - This provides empirical proof of conditional heteroscedasticity, establishing the rationale for GARCH volatility modelling in Phase 6.

5. **Distributional Non-Normality**:
   - `cooling_kw` is strongly right-skewed ($S = +2.20$) and leptokurtic ($K = +4.71$). Linear models assuming Gaussian errors will underestimate tail risk, reinforcing the need for GARCH with Student-t or skewed innovation distributions.